# 第 7 章 · softmax 饱和

配套讲义：[在线版](https://wikieden.github.io/transformer-visual/lessons/07-saturation.html) · 仓库里 `transformer-deep-dive/notes/lessons/07-saturation.md`

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 torch 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import torch

torch.set_printoptions(precision=3, sci_mode=False)  # 打印保留 3 位小数，不用科学计数法
print("torch", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 / 说法 | PyTorch 写法 | 跑出来 |
|---|---|---|---|
| 放大过头 | [2, 1, 50] | `torch.softmax` | [0, 0, 1] |
| 调高 1 分 | 5 → 6；50 → 51 | 改分数再 softmax | 0.936 → 0.976；1.000 → 1.000（没变） |
| 学不动 = 斜率是 0 | 调一点，比例变多少 | `torch.softmax(s, -1)[2].backward()` | 5 分时 0.0597；50 分时 0 |
| 另一种过头：溢出 | $e^{100}$ | `torch.exp(100.)` | inf，手写 softmax 得 nan |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import torch

**函数 `nudge`**：把第 idx 个分数调高 1 分，看它的比例前后各是多少。

In [ ]:
def nudge(scores: list[float], idx: int = 2) -> tuple[float, float]:
    """把第 idx 个分数调高 1 分，看它的比例前后各是多少。"""
    s = torch.tensor(scores)
    before = float(torch.softmax(s, -1)[idx])
    s[idx] += 1
    after = float(torch.softmax(s, -1)[idx])
    return before, after

**函数 `slope`**："调一点点，比例变多少"：用 autograd 算 ∂p_idx / ∂s_idx（等于 p(1 − p)）。

In [ ]:
def slope(scores: list[float], idx: int = 2) -> float:
    """ "调一点点，比例变多少"：用 autograd 算 ∂p_idx / ∂s_idx（等于 p(1 − p)）。"""
    s = torch.tensor(scores, requires_grad=True)
    torch.softmax(s, -1)[idx].backward()
    return float(s.grad[idx])

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("[2, 1, 50] 的 softmax:", [round(float(v), 4) for v in torch.softmax(torch.tensor([2.0, 1, 50]), -1)])
for scores in ([2.0, 1.0, 5.0], [2.0, 1.0, 50.0]):
    b, a = nudge(scores)
    print(
        f"小红 {scores[2]:.0f} → {scores[2] + 1:.0f}：{b:.3f} → {a:.3f}，变了 {a - b:.4f}；斜率 {slope(scores):.2e}"
    )
print("斜率 ≈ 0 就是'学不动'：训练时靠这个斜率（梯度）决定往哪调，它没了，就没有学习信号。")

In [ ]:
print("\n代码里的另一种'放大过头'：e^x 太大，float32 装不下")
big = torch.tensor([2.0, 1.0, 100.0])
e = torch.exp(big)
print("  torch.exp([2, 1, 100]) =", [round(float(v), 3) for v in e])
print("  手写 e / e.sum()        =", (e / e.sum()).tolist(), "← inf / inf = nan")
print(
    "  torch.softmax          =",
    [f"{float(v):.3g}" for v in torch.softmax(big, -1)],
    "← 内部先减掉最大值，不会溢出",
)
shifted = big - big.max()
print(
    "  减最大值后再算         =",
    [f"{float(v):.3g}" for v in torch.softmax(shifted, -1)],
    "（结果不变：分子分母同除 e^max）",
)

## 3. 代码里才看得到的

1. **饱和就是斜率（梯度）≈ 0**：用第 5 章的 `backward()` 直接算"调高一点，比例变多少"。5 分时还有 0.06，50 分时正好是 0。训练是照着这个斜率调参数的，它没了就学不动。
2. **float32 最大只能表示约 $3.4 \times 10^{38}$**，$e^{89}$ 就超了，变成 inf（infinity：无穷大）。手写的 `e / e.sum()` 变成 inf ÷ inf = nan。
3. **`torch.softmax` 内部先减掉最大值**：分子分母同时除以 $e^{\max}$，结果不变，但最大的那个变成 $e^0 = 1$，永远不会溢出。这就是第 6 章说"不要手写"的原因。

## 4. 自己改一改

- 把上面任意一格里的数字改一改，再按 Shift + Enter，看结果怎么变。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 分数太大，比例推到 [0, 0, 1]；再调分数纹丝不动，这叫饱和，学不动。
> 饱和是学不进，过拟合是背过头，两回事。